# 04. 데이터셋 생성
주가 피처 + 감성 점수를 합쳐 시계열 입력 데이터셋 생성

- `news_count`를 sentiment_mean/std에 가중치로 반영 (직접 피처 X)
- 감성 파생 피처: sentiment_lag, sentiment_change, news_count_zscore_20
- 종목별 train/test split 후 train 기준 Min-Max 정규화
- 종목별 시간 순서를 유지한 채 시퀀스 생성

In [ ]:
# Google Drive 마운트 (Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 필요 패키지 설치
!pip install -q scikit-learn

In [ ]:
# ── 설정 ──────────────────────────────────────────────────────────────────────
DATA_DIR    = "/content/drive/MyDrive/term_project/data"
WINDOW_SIZE = 20
SEED        = 42

# 뉴스 데이터 기간 기준 고정 split
NEWS_START_DATE = "2025-04-06"   # sentiment.csv 시작 날짜
NEWS_END_DATE   = "2026-04-10"   # sentiment.csv 종료 날짜
TRAIN_MONTHS    = 10             # 앞 10개월 → train / 나머지 2개월 → test

import os
os.makedirs(DATA_DIR, exist_ok=True)
print(f"DATA_DIR : {DATA_DIR}")
print(f"Train 구간: {NEWS_START_DATE} ~ (+ {TRAIN_MONTHS}개월)")
print(f"Test  구간: (+ {TRAIN_MONTHS}개월) ~ {NEWS_END_DATE}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# ── 피처 컬럼 정의 ────────────────────────────────────────────────────────────
FEATURE_COLS = [
    # 기술적 지표
    "log_return", "volume",
    "sma_5", "sma_20", "sma_60",
    "rsi", "macd", "macd_signal", "macd_hist",
    "bb_upper", "bb_lower", "bb_width",

    # 모멘텀
    "return_5d", "return_20d",

    # 거래량 이상
    "volume_ratio_20",

    # 52주 고가 대비 위치
    "high_52w_ratio",

    # 변동성
    "volatility_20",

    # 시장 대비 초과 수익률
    "relative_return",

    # 추세 강도 (ADX)
    "adx",

    # 거래량+가격 방향 결합 (OBV)
    "obv",

    # news_count 기반 weighted sentiment
    "sentiment_mean_weighted",
    "sentiment_std_weighted",

    # 감성 시계열 파생 피처
    "sentiment_lag1",
    "sentiment_lag2",
    "sentiment_change",
    "news_count_zscore_20",

    # 감성 방향성 정교화: 연속형 신호 (binary 플래그 대체)
    # - sentiment_up_signal   = max(sentiment_mean, 0) * log1p(news_count)
    #   긍정 감성이 강하고 뉴스가 많을수록 큰 값 (binary >0.1 보다 풍부한 정보)
    # - sentiment_down_signal = max(-sentiment_mean, 0) * log1p(news_count)
    #   부정 감성이 강하고 뉴스가 많을수록 큰 값
    "sentiment_up_signal",
    "sentiment_down_signal",
]

print(f"피처 수: {len(FEATURE_COLS)}")

In [ ]:
# ── 헬퍼 함수 ─────────────────────────────────────────────────────────────────
def create_sequences(df: pd.DataFrame, window: int, feature_cols: list,
                     with_meta: bool = False):
    """시계열 윈도우 데이터 생성.
    with_meta=True 이면 각 샘플의 (code, date) 메타데이터도 반환.
    """
    X, y, meta = [], [], []

    data   = df[feature_cols].values
    target = df["target"].values

    for i in range(window, len(data)):
        X.append(data[i - window:i])
        y.append(target[i])
        if with_meta:
            meta.append((df["code"].iloc[i], str(df["date"].iloc[i])[:10]))

    if with_meta:
        return np.array(X), np.array(y), meta
    return np.array(X), np.array(y)


def add_weighted_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    """
    news_count를 감성 피처에 가중치로 반영.

    [기존] sentiment_mean_weighted, sentiment_std_weighted
    [추가] sentiment_up_signal, sentiment_down_signal
        binary 플래그(sentiment_positive/negative) 대신 연속형으로 교체:
        - sentiment_up_signal   = max(sentiment_mean, 0) * log1p(news_count)
          → 긍정 감성이 강하고 뉴스가 많을수록 큰 값
        - sentiment_down_signal = max(-sentiment_mean, 0) * log1p(news_count)
          → 부정 감성이 강하고 뉴스가 많을수록 큰 값
    news_count=0 이면 log1p(0)=0 이므로 감성 피처도 자동으로 0이 됨.
    """
    df = df.copy()
    weight = np.log1p(df["news_count"])

    df["sentiment_mean_weighted"] = df["sentiment_mean"] * weight
    df["sentiment_std_weighted"]  = df["sentiment_std"]  * weight

    # 연속형 방향성 감성 피처 (binary 플래그 대체)
    df["sentiment_up_signal"]   = np.clip(df["sentiment_mean"],  0, None) * weight
    df["sentiment_down_signal"] = np.clip(-df["sentiment_mean"], 0, None) * weight

    return df

In [ ]:
# ── 데이터 로드 & 병합 ────────────────────────────────────────────────────────
features_path  = os.path.join(DATA_DIR, "features.csv")
sentiment_path = os.path.join(DATA_DIR, "sentiment.csv")

df_feat = pd.read_csv(features_path, parse_dates=["date"])
df_feat["code"] = df_feat["code"].astype(str).str.zfill(6)

df_sent = pd.read_csv(sentiment_path, parse_dates=["date"])
df_sent.rename(columns={"종목코드": "code"}, inplace=True)
df_sent["code"] = df_sent["code"].astype(str).str.zfill(6)

df = df_feat.merge(
    df_sent,
    on=["code", "date"],
    how="left",
    validate="one_to_one"
)

print(f"병합 데이터: {len(df)}행, 종목 수: {df['code'].nunique()}")
df.head()

In [ ]:
# ── 감성 피처 처리 ────────────────────────────────────────────────────────────
sentiment_cols = [
    "sentiment_mean",
    "sentiment_std",
    "news_count",
    "sentiment_lag1",
    "sentiment_lag2",
    "sentiment_change",
    "news_count_zscore_20",
]

# 뉴스가 없는 날 감성 피처를 0으로 처리
for col in sentiment_cols:
    if col not in df.columns:
        df[col] = 0

df[sentiment_cols] = df[sentiment_cols].fillna(0)

# news_count를 직접 피처로 쓰지 않고 감성 피처에 반영
# add_weighted_sentiment 내부에서 sentiment_up_signal / sentiment_down_signal 도 함께 생성됨
df = add_weighted_sentiment(df)

# 모델 입력에 필요한 컬럼 결측 제거
df = df.dropna(subset=FEATURE_COLS + ["target"])

print(f"결측 제거 후: {len(df)}행")
print(f"sentiment_up_signal   > 0 비율: {(df['sentiment_up_signal']   > 0).mean():.1%}")
print(f"sentiment_down_signal > 0 비율: {(df['sentiment_down_signal'] > 0).mean():.1%}")
print(f"sentiment_up_signal   평균: {df['sentiment_up_signal'].mean():.4f}")
print(f"sentiment_down_signal 평균: {df['sentiment_down_signal'].mean():.4f}")

In [ ]:
# ── 종목별 시퀀스 생성 ────────────────────────────────────────────────────────
np.random.seed(SEED)

NEWS_START   = pd.Timestamp(NEWS_START_DATE)
NEWS_END     = pd.Timestamp(NEWS_END_DATE)
TRAIN_CUTOFF = NEWS_START + pd.DateOffset(months=TRAIN_MONTHS)

print(f"Train 구간: {NEWS_START.date()} ~ {TRAIN_CUTOFF.date()}")
print(f"Test  구간: {TRAIN_CUTOFF.date()} ~ {NEWS_END.date()}")

all_X_train, all_y_train = [], []
all_X_test,  all_y_test  = [], []
all_train_meta, all_test_meta = [], []

used_codes    = 0
skipped_codes = 0

for code, group in df.groupby("code"):
    group = group.sort_values("date").reset_index(drop=True)

    context_start = NEWS_START - pd.Timedelta(days=40)
    group = group[(group["date"] >= context_start) & (group["date"] <= NEWS_END)].copy()

    if len(group) < WINDOW_SIZE + 2:
        skipped_codes += 1
        continue

    train_group  = group[group["date"] < TRAIN_CUTOFF].copy()

    test_context = train_group.iloc[-WINDOW_SIZE:].copy()
    test_only    = group[group["date"] >= TRAIN_CUTOFF].copy()
    test_group   = pd.concat([test_context, test_only]).reset_index(drop=True)

    if len(train_group) < WINDOW_SIZE + 1 or len(test_group) < WINDOW_SIZE + 1:
        skipped_codes += 1
        continue

    scaler = MinMaxScaler()
    train_group[FEATURE_COLS] = scaler.fit_transform(train_group[FEATURE_COLS])
    test_group[FEATURE_COLS]  = scaler.transform(test_group[FEATURE_COLS])

    X_train_part, y_train_part, train_meta = create_sequences(
        train_group, WINDOW_SIZE, FEATURE_COLS, with_meta=True
    )
    X_test_part, y_test_part, test_meta = create_sequences(
        test_group, WINDOW_SIZE, FEATURE_COLS, with_meta=True
    )

    all_X_train.append(X_train_part)
    all_y_train.append(y_train_part)
    all_X_test.append(X_test_part)
    all_y_test.append(y_test_part)
    all_train_meta.extend(train_meta)
    all_test_meta.extend(test_meta)

    used_codes += 1

if not all_X_train or not all_X_test:
    raise ValueError("No train/test sequences were generated. Check data size or WINDOW_SIZE.")

X_train = np.concatenate(all_X_train)
y_train = np.concatenate(all_y_train)
X_test  = np.concatenate(all_X_test)
y_test  = np.concatenate(all_y_test)

train_codes = np.array([m[0] for m in all_train_meta])
train_dates = np.array([m[1] for m in all_train_meta])
test_codes  = np.array([m[0] for m in all_test_meta])
test_dates  = np.array([m[1] for m in all_test_meta])

print(f"사용 종목 수: {used_codes}, 제외 종목 수: {skipped_codes}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
# ── 저장 ──────────────────────────────────────────────────────────────────────
output_path = os.path.join(DATA_DIR, "dataset9.npz")

np.savez(
    output_path,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=np.array(FEATURE_COLS),
    train_codes=train_codes,
    train_dates=train_dates,
    test_codes=test_codes,
    test_dates=test_dates,
)

print(f"저장 완료: {output_path}")
print(f"Train: {X_train.shape}  |  train_dates: {train_dates.shape}")
print(f"Test : {X_test.shape}   |  test_dates : {test_dates.shape}")
# numpy 문자열 배열은 .min()/.max() ufunc 미지원 → Python 내장 min/max 사용
print(f"train 날짜 범위: {min(train_dates)} ~ {max(train_dates)}")
print(f"test  날짜 범위: {min(test_dates)} ~ {max(test_dates)}")